# TFPARN (Transformer-based Focal-Pairwise Attentive Ranking Network) for Anti-Spoofing: Complete Technical Documentation

This notebook documents **TFPARN** and its application to the **ASVspoof 5 Track 1 closed condition** (audio anti-spoofing / speech deepfake detection). The system distinguishes genuine human speech (*bonafide*) from AI-generated synthetic speech (*spoof*).

It accompanies the paper:

> **A Training-Efficient Transformer-Based Anti-Spoofing Network for Logical Access in ASVspoof 5** — Sidan Yin (San Domenico School) and Bo Zhao (University of Washington).

**Central thesis.** TFPARN is designed *not* to chase the single best detection score, but to strike a favorable **balance between detection performance and computational cost** (training *and* inference). It encodes log-Mel frames with a Transformer, aggregates them with attention pooling, and trains with a combined **focal + pairwise-ranking** objective that aligns optimization with the ranking/threshold-based challenge metrics (EER, minDCF).

**Headline results** (mean over seeds 42/63/2026, vs. re-implemented baselines under one protocol):

| | TFPARN (full) | AASIST | RawNet2 |
|---|---|---|---|
| minDCF (primary) ↓ | **0.2430** | 0.2911 | 0.5375 |
| EER (%) ↓ | **12.52** | 18.58 | 27.23 |
| Inference memory ↓ | **1.4 GB** | 56.7 GB | 4.9 GB |
| Latency / utt ↓ | **0.79 ms** | 10.48 ms | 0.78 ms |

TFPARN wins on detection while using the least inference memory and reaching its best checkpoint in far less training time than AASIST. Full tables and the ablation chain are in Section 10.

## 1. Background

### 1.1 ASVspoof 5 Track 1 Overview

The **ASVspoof** challenge series develops general-purpose countermeasures against synthetic and manipulated speech. The latest edition, **ASVspoof 5**, has two tracks; this work targets **Track 1** (stand-alone spoofing / speech deepfake detection), under the **closed condition** — only the official ASVspoof 5 training partition may be used, with **no external data and no pre-trained models**.

A Track 1 system receives telephony/VoIP speech and outputs a single real-valued **detection score**, where larger values indicate stronger support for the *bonafide* class:
- **Bonafide (Label = 1):** genuine human speech
- **Spoof (Label = 0):** AI-generated / manipulated speech

**Data (Track 1).** ASVspoof 5 is built from the English subset of Multilingual LibriSpeech (MLS). The train/dev/eval partitions have **non-overlapping speakers and spoofing attacks**, which tests generalization to unseen conditions:

| Subset | Speakers | Attacks | Utterances | Spoof | Bonafide |
|--------|---------:|--------:|-----------:|------:|---------:|
| Train  | 400      | 8       | 182,357    | 163,560 | 18,797 |
| Dev    | 785      | 8       | 140,950    | 109,616 | 31,334 |

The eval set uses a further-disjoint set of 16 attacks. Note the strong **class imbalance** (≈8.7:1 spoof:bonafide in train) and the speaker/attack disjointness across splits.

### 1.2 Why TFPARN? (Motivation)

Track 1 poses three difficulties that motivate TFPARN's design:

1. **Hard trials are under-emphasized by cross-entropy.** As training proceeds most samples become easy, and standard CE gives too little weight to the smaller set of confusing, near-boundary utterances → **focal loss** keeps optimization focused on hard samples.
2. **The metrics are ranking/threshold-based, not accuracy-based.** EER and minDCF assess whether bonafide scores are *globally* higher than spoof scores; CE does not enforce a favorable global ordering → a **pairwise ranking loss** is added to align training with EER/minDCF.
3. **Spoofing traces are local and sparse** in time and frequency, so naive global aggregation dilutes the most informative regions → **attention pooling** emphasizes the frames most likely to carry spoofing cues.

On top of detection quality, **deployment efficiency matters**: ASV devices have long service lives, limited hardware, low-latency requirements, and must be periodically retrained as new data arrive. So TFPARN is explicitly evaluated on *both* detection performance and **training/inference cost**.

### 1.3 Contributions

- A Transformer-based anti-spoofing architecture for the ASVspoof 5 Track 1 closed condition, designed to **balance detection performance against training and inference cost**.
- **Attentive temporal pooling** (local-cue focus) + **focal loss** (hard-trial emphasis) + **pairwise ranking loss** (alignment with ranking-oriented metrics).
- A comparison with re-implemented **AASIST** and **RawNet2** baselines under a unified protocol, on both detection metrics *and* computational efficiency.

### 1.4 Evaluation Metrics

ASVspoof 5 Track 1 reports **minDCF (primary)**, **EER**, **Cllr**, and **actDCF**. For Track 1, minDCF/actDCF use spoof prior `π_spf = 0.05` with costs `C_miss = 1` and `C_fa = 10`. Lower is better for all four. See Section 6 for the exact definitions used in the code.

## 2. Environment Setup

### 2.1 Dependencies

The repository ships **two** requirement files for two separate purposes:

| File | Purpose | Key packages |
|------|---------|--------------|
| `Model/requirements.txt` | Training & evaluation | `torch==2.11.0`, `torchaudio==2.11.0`, `numpy==2.4.4`, `scikit-learn==1.7.2`, `scipy==1.15.3`, `soundfile==0.13.1`, `tqdm==4.67.3` (plus the CUDA 13 `nvidia-*` wheels) |
| `CsvAnalyzer/requirements.txt` | Plotting / analysis | `matplotlib==3.10.8`, `SciencePlots==2.2.1`, `numpy==2.4.4`, Jupyter (`ipykernel`, `ipython`) |

**Installation:**
```bash
# For training / evaluating models
pip install -r Model/requirements.txt

# For generating the training-efficiency plots
pip install -r CsvAnalyzer/requirements.txt
```

**Python Version:** 3.10 or higher.

### 2.2 Hardware Requirements

**Recommended Environment** (sized to run *everything* in this repo, including the AASIST baseline):
- **OS:** Linux
- **CPU:** 16-core processor or higher
- **RAM:** 128GB or more
- **Storage:** 500GB or more of free disk space
- **GPU:** A CUDA-capable NVIDIA GPU with 90GB+ VRAM (developed and tested on an **RTX Pro 6000**)

**Per-model VRAM:** TFPARN itself is light. As a rough footprint guide:

- **Batch size 64:** ~8GB VRAM (the default)
- **Batch size 96:** ~10GB VRAM
- **Batch size 128:** ~12GB VRAM

TFPARN therefore runs on a single 8GB+ GPU — or even **CPU-only** (much slower). The 90GB+ recommendation above is driven by the **AASIST baseline**, which is very memory-hungry and needs ~80–90GB VRAM at `batch_size=64`; TFPARN and RawNet2 are far more modest. Lower `batch_size` / `num_workers` to fit a smaller GPU.

## 3. Model Architecture

### 3.1 Complete Pipeline Overview

The model follows a complete Transformer-based architecture (`SpeechTransformerClassifier` in `Model/model.py`):

```
Raw Waveform -> Log-Mel Spectrogram -> Transformer Encoder -> Pooling -> Classification
```

**Architecture Stages:**

1. **Frontend:** Log-Mel Spectrogram extraction (computed in-model from the raw waveform)
2. **Embedding:** LayerNorm **along the frequency dimension** + Linear projection to `d_model`
3. **Positional Encoding:** Sinusoidal positional embeddings
4. **Backbone:** Multi-layer Transformer Encoder (self-attention)
5. **Pooling:** Mean / Attention / Top-k pooling with masking (Attention pooling is the TFPARN default)
6. **Classification Head:** 2-layer MLP -> Binary logits (`o₀` = spoof, `o₁` = bonafide)

For a 4.0 s input at 16 kHz with `n_fft=1024` and `hop_length=160`, the log-Mel sequence has **T′ ≈ 401 frames** of `F = n_mels = 160` bands, so the encoder operates on a compact `401 × 256` sequence (after projection to `d_model=256`). This short sequence is the key reason TFPARN is cheap to run — self-attention is quadratic in sequence length, but 401 frames keep that cost small.

### 3.2 Model Configuration

The architecture is configured by the `SpeechClassifierArgs` dataclass in `Model/model.py`. Its defaults are:

```python
from dataclasses import dataclass

@dataclass
class SpeechClassifierArgs:
    # Mel Spectrogram parameters
    n_mels: int = 128          # Number of mel filterbanks
    n_fft: int = 768           # FFT window size
    hop_length: int = 160      # Hop length for STFT
    sample_rate: int = 16000   # Audio sample rate

    # Transformer parameters
    d_model: int = 256          # Model dimension
    nhead: int = 8              # Number of attention heads
    num_layers: int = 6         # Number of Transformer layers
    dim_feedforward: int = 1024 # FFN hidden dimension
    dropout: float = 0.3        # Dropout probability
    activation: str = "relu"    # Activation function

    # Pooling method: "mean", "attention", "top-k"
    pooling_method: str = "mean"
    top_k_ratio: float = 0.3    # For top-k pooling
```

> **Note:** These are the bare `model.py` defaults. When training through `Model/main_train.py`, the `ModelArgs` config builds a `SpeechClassifierArgs` and **overrides** several of these — the TFPARN configuration in the paper uses `n_mels=160`, `n_fft=1024`, `hop_length=160`, `d_model=256`, `num_layers=6` (L=6), `nhead=8` (h=8), `dim_feedforward=1024` (d_ff), `dropout=0.3`, and `pooling_method="attention"`. See Section 5.1.

### 3.3 Architecture Code Example

```python
from model import create_model, SpeechClassifierArgs

# Create model with default configuration
args = SpeechClassifierArgs()
model = create_model(args)

# Or customize the architecture (the TFPARN configuration)
args = SpeechClassifierArgs(
    n_mels=160,
    n_fft=1024,
    d_model=256,
    num_layers=6,
    pooling_method="attention"
)
model = create_model(args)

# Move to device
model = model.to(device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_params:,}")
```

### 3.4 Key Architecture Features

**1. In-Model Mel Spectrogram Computation:**
- No need for preprocessing
- The mel filterbank and Hann window are registered as buffers (not trainable)
- Consistent processing during training and inference

**2. Positional Encoding:**
```python
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Sinusoidal positional encoding
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe)
```

**3. Flexible Pooling Strategies:**

- **Mean Pooling:** masked average of all frame embeddings (the ablation variant in the paper)
- **Attention Pooling:** learned attention weights for aggregation (the TFPARN default)
- **Top-k Pooling:** select the `top_k_ratio` fraction of frames by L2 norm

**4. Classification Head:**
```python
self.classifier = nn.Sequential(
    nn.Linear(d_model, d_model // 2),
    nn.ReLU(),
    nn.Dropout(dropout),
    nn.Linear(d_model // 2, 2)  # Binary: [spoof, bonafide]
)
```

### 3.5 Forward Pass

```python
# Input: [B, 1, T] raw waveform (mono)
# Output: [B, 2] logits [spoof_score, bonafide_score]

waveforms = batch['waveforms'].to(device)  # [B, 1, 64000]
logits = model(waveforms)  # [B, 2]

# Get predictions
probs = torch.softmax(logits, dim=1)
predictions = torch.argmax(logits, dim=1)  # 0=spoof, 1=bonafide
```

### 3.6 Attention Pooling and Parameter Count

**Attention pooling** (default) scores each encoded frame `Hₜ` with a 2-layer perceptron, normalizes the scores with softmax over the *valid* frames (padding positions are set to `-∞` before softmax), and takes the weighted sum:

```
eₜ = w₂ᵀ · tanh(W₁ Hₜ + b₁) + b₂          # W₁: 256→128,  w₂: 128→1
αₜ = softmax(eₜ)  over t = 1 … T′
h_utt = Σₜ αₜ Hₜ                            # utterance embedding ∈ ℝ²⁵⁶
```

The **mean-pooling ablation** simply replaces this with a masked average over the 401 valid frames (no trainable parameters).

**Backbone tensor shapes & trainable parameters** (single utterance, TFPARN config):

| Module / Layer | Output size | Trainable params |
|---|---|---:|
| Input LayerNorm (over 160 mel bands) | 160 | 320 |
| Feature projection Linear(160 → 256) | 401 × 256 | 41,216 |
| Transformer Encoder × 6 (heads=8, FFN=1024) | 401 × 256 | 4,738,560 |
| Attention pooling (256 → 128 → 1) | 256 | 33,025 |
| Classifier: Linear(256 → 128) + Linear(128 → 2) | 2 | 33,154 |
| **Total — attention pooling (default)** | | **4,846,275** (≈4.85M) |
| **Total — mean pooling (ablation)** | | **4,813,250** (≈4.81M) |

So attention pooling adds only ~33k parameters over the mean-pooling variant — a negligible cost for a consistent accuracy gain (Section 10).

## 4. Data Processing Pipeline

### 4.1 Data Loading Overview

The data pipeline (`Model/data_process.py`) handles:
- Protocol file parsing (`read_protocol`)
- Audio loading (FLAC, via `soundfile`)
- Fixed-length processing (crop / repeat strategy)
- RawBoost augmentation (training only)
- Test-Time Augmentation (dev / eval, via `TTADataset`)

### 4.2 Protocol File Format

ASVspoof5 protocol files contain 10 whitespace-separated columns:

```
speaker_id  file_name  gender  codec  codec_q  codec_seed  attack_tag  attack_label  KEY  tmp
```

**Label Mapping** (from the `KEY` field):
- `bonafide` → 1 (genuine human speech)
- `spoof` → 0 (AI-generated speech)

### 4.3 Audio Processing Strategy

**Fixed-Length Processing:**
```python
# Target duration: 4.0 seconds at 16 kHz = 64,000 samples
duration_sec = 4.0
sample_rate = 16000
target_length = int(duration_sec * sample_rate)  # 64,000

# If audio is longer: crop (random for train, center for dev/eval)
# If audio is shorter: repeat-concatenate, then crop to target_length
```

**Normalization:**
- Convert to mono (if stereo)
- Resample to 16 kHz (if needed)
- Normalize amplitude to [-1, 1]

### 4.4 RawBoost Data Augmentation

`RawBoost` randomly picks one of three algorithms per call (training only, applied with `rawboost_prob`):

**Algorithm 1 — Linear/Nonlinear Convolution**
```python
N_fir = np.random.randint(5, 15)
h = np.random.randn(N_fir); h = h / np.sum(np.abs(h))
x_conv = signal.convolve(x, h, mode='same')
if np.random.rand() > 0.5:                       # optional nonlinear distortion
    x_conv = np.tanh(np.random.uniform(0.1, 0.5) * x_conv)
```

**Algorithm 2 — IIR Filtering** (random lowpass / highpass / bandpass)
```python
cutoff = np.random.uniform(1000, 4000)           # e.g. lowpass
b, a = signal.butter(4, cutoff / (sample_rate / 2), btype='low')
x_filtered = signal.filtfilt(b, a, x)
```

**Algorithm 3 — Stationary Additive Noise** (random SNR 10–40 dB)
```python
snr_db = np.random.uniform(10, 40)
signal_power = np.mean(x ** 2)
noise_power = signal_power / (10 ** (snr_db / 10))
x_noisy = x + np.random.randn(len(x)) * np.sqrt(noise_power / np.mean(noise ** 2))
```

### 4.5 DataLoader Creation

`make_loaders` parses the three protocols, builds the datasets (wrapping dev/eval with `TTADataset` when `use_tta=True`), and returns the three loaders.

```python
from data_process import make_loaders, DefaultArgs

args = DefaultArgs()
args.train_data_dir = "path/to/train/flac/"
args.train_protocol_dir = "path/to/train.tsv"
# ... (dev/eval paths) ...
args.batch_size = 64            # default
args.num_workers = 32           # default
args.use_rawboost = True        # RawBoost for training
args.use_tta = True             # TTA for dev/eval
args.tta_num_crops = 5

train_loader, dev_loader, eval_loader = make_loaders(args)

# Train batches (no TTA): waveforms are [B, 1, 64000]
for batch in train_loader:
    waveforms = batch['waveforms']  # [B, 1, 64000]
    labels    = batch['labels']     # [B]
    break

# Dev/Eval batches (with TTA): waveforms are [B, num_crops, 1, 64000]
for batch in dev_loader:
    waveforms = batch['waveforms']  # [B, 5, 1, 64000]
    break
```

## 5. Training Pipeline

### 5.1 Training Configuration

Training is driven by the `ModelArgs` dataclass in `Model/main_train.py`. It holds the data, model, and training settings in one place and builds the `DefaultArgs` (data) and `SpeechClassifierArgs` (model) objects internally. The TFPARN defaults are:

```python
from dataclasses import dataclass

@dataclass
class ModelArgs:
    # Data / protocol paths (empty by default — must be filled in)
    train_data_dir: str = ""
    dev_data_dir: str = ""
    eval_data_dir: str = ""
    train_protocol_dir: str = ""
    dev_protocol_dir: str = ""
    eval_protocol_dir: str = ""

    # Audio + DataLoader
    sample_rate: int = 16000
    duration_sec: float = 4.0
    batch_size: int = 64
    num_workers: int = 32
    use_rawboost: bool = True       # RawBoost augmentation (training only)
    rawboost_prob: float = 0.5
    use_tta: bool = True            # Test-Time Augmentation for dev/eval
    tta_num_crops: int = 5

    # Model architecture (-> SpeechClassifierArgs)
    n_mels: int = 160
    n_fft: int = 1024
    hop_length: int = 160
    d_model: int = 256
    nhead: int = 8
    num_layers: int = 6
    dim_feedforward: int = 1024
    model_dropout: float = 0.3
    activation: str = "relu"
    pooling_method: str = "attention"   # "mean", "attention", or "top-k"
    top_k_ratio: float = 0.3

    # Training hyperparameters
    max_epochs: int = 100
    learning_rate: float = 0.667e-4
    weight_decay: float = 1e-2
    optimizer_type: str = "adamw"       # "adam" or "adamw"
    scheduler_type: str = "cosine"      # "cosine", "step", or "none"
    scheduler_warmup_epochs: int = 5

    # Loss function
    loss_type: str = "focal"            # "ce" or "focal"
    focal_alpha: float = 0.5            # bonafide weight (spoof gets 1 - alpha) → equal weights
    focal_gamma: float = 2.0            # focusing parameter

    # Pairwise ranking loss
    enable_pairwise: bool = True
    pairwise_margin: float = 1.0        # m
    pairwise_weight: float = 0.3        # λ

    # Early stopping
    early_stopping_patience: int = 15
    early_stopping_metric: str = "min_dcf"  # 'eer', 'min_dcf', 'f1_macro', 'accuracy', ...
    early_stopping_mode: str = "min"        # 'min' for eer/min_dcf, 'max' for f1/acc/recall/auc

    # Checkpoint dir (empty by default — must be filled in)
    save_dir: str = ""
    seed: int = 42
```

> **Paper note on the learning rate.** The paper specifies a base learning rate of `1e-4` and applies the **linear scaling rule** `η = η_ref × 64 / B_ref` so that all compared systems are trained fairly at the same batch size (64). The code default above (`0.667e-4`) is the scaled value for this setup; adjust it if you change `batch_size`.

### 5.2 Focal Classification Loss

Focal loss (in `Model/utils.py`) keeps optimization focused on the small set of **hard, near-boundary trials** that cross-entropy under-emphasizes once most samples are easy:

```python
class FocalLoss(nn.Module):
    """
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Args:
        alpha: Weighting factor [num_classes]
        gamma: Focusing parameter (default: 2.0)
    """
    def __init__(self, alpha: torch.Tensor, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, labels):
        probs = F.softmax(logits, dim=1)
        probs = torch.clamp(probs, min=1e-7, max=1.0 - 1e-7)
        labels_one_hot = F.one_hot(labels, num_classes=logits.shape[1]).float()
        probs_t = (probs * labels_one_hot).sum(dim=1)

        alpha_t = (self.alpha.to(logits.device) * labels_one_hot).sum(dim=1)
        focal_weight = alpha_t * torch.clamp((1 - probs_t) ** self.gamma, min=1e-7)
        ce_loss = F.cross_entropy(logits, labels, reduction='none')

        return (focal_weight * ce_loss).mean()

# main_train.py turns the scalar focal_alpha into a [spoof, bonafide] weight vector:
#   focal_alpha = torch.tensor([1.0 - args.focal_alpha, args.focal_alpha])  # = [0.5, 0.5]
criterion = FocalLoss(focal_alpha, gamma=2.0)
```

**Why focal loss — and why α = 0.5?** Although the ASVspoof 5 training set is imbalanced (≈8.7:1 spoof:bonafide), TFPARN deliberately sets **α = 0.5 (equal class weights)**. Focal loss is therefore used here **only to emphasize hard, ambiguous utterances** via the `(1 − p_t)^γ` factor (`γ = 2.0`), **not** to rebalance the classes — no class resampling or re-weighting is applied during training. In the ablation this is the single change that most improves the calibration-dependent Cllr (Section 10).

### 5.3 Pairwise Ranking Loss

EER and minDCF are *ranking/threshold-sensitive*: they care whether bonafide scores are globally higher than spoof scores, not per-sample accuracy. The pairwise ranking loss (combined via `CombinedLoss`) directly targets this. The bonafide-class logit `z₁` is used as the detection score `s(x)`:

```python
class PairwiseRankingLoss(nn.Module):
    """
    Encourages bonafide samples to score higher than spoof samples
    Loss = max(0, margin - (score_bonafide - score_spoof))
    """
    def __init__(self, margin: float = 1.0):
        super().__init__()
        self.margin = margin

    def forward(self, logits, labels):
        scores = logits[:, 1]  # bonafide-class logit = detection score s(x)

        bonafide_scores = scores[labels == 1]   # set B
        spoof_scores = scores[labels == 0]       # set S
        if len(bonafide_scores) == 0 or len(spoof_scores) == 0:
            return torch.tensor(0.0, device=logits.device)

        # All (b, s) pairs -> hinge over the margin, averaged over |B|*|S| pairs
        score_diff = bonafide_scores[:, None] - spoof_scores[None, :]
        return F.relu(self.margin - score_diff).mean()

# Combined loss (CombinedLoss): L_TFPARN = L_focal + λ · L_pair,  with m=1.0, λ=0.3
total_loss = focal_loss + 0.3 * pairwise_loss
```

The combined objective is `L_TFPARN = L_focal + λ·L_pair` with margin `m = 1.0` and weight `λ = 0.3`. (The paper notes a limitation: this in-batch hinge only loosely approximates minDCF/actDCF; a listwise/differentiable ranking loss could track the metrics more directly.)

### 5.4 Optimizer and Scheduler

```python
# AdamW optimizer with weight decay
optimizer = optim.AdamW(model.parameters(), lr=0.667e-4, weight_decay=1e-2)

# Cosine annealing scheduler with linear warmup
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max_epochs - warmup_epochs,
    eta_min=1e-6
)

# Linear warmup for the first 5 epochs
for epoch in range(1, warmup_epochs + 1):
    warmup_lr = learning_rate * epoch / warmup_epochs
    for param_group in optimizer.param_groups:
        param_group['lr'] = warmup_lr
```

The first 5 epochs warm up linearly to the base LR, then cosine annealing decays it toward `1e-6` — avoiding large early gradient oscillations while allowing fine adjustments later.

### 5.5 Training Loop

The loop trains one epoch, validates on Dev **with TTA**, selects the best epoch by `early_stopping_metric` (Dev **minDCF** by default), and saves a rolling `best_model.pt`. Early stopping triggers if Dev minDCF does not improve (tolerance ≈ `1e-4`) for 15 consecutive epochs. Per-epoch time, peak memory, and metrics are logged for the CSV analyzer.

```python
from utils import EarlyStopping, compute_all_metrics

early_stopping = EarlyStopping(patience=15, mode='min')  # 'min' for minDCF
best_metric = float('inf')

for epoch in range(1, max_epochs + 1):
    # --- Train ---
    model.train()
    for batch in train_loader:
        waveforms = batch['waveforms'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        loss = criterion(model(waveforms), labels)
        loss.backward()
        optimizer.step()

    # --- Validate (with TTA) ---
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in dev_loader:
            waveforms = batch['waveforms'].to(device)  # [B, num_crops, C, T]
            labels = batch['labels'].to(device)
            B, num_crops, C, T = waveforms.shape
            logits_flat = model(waveforms.view(B * num_crops, C, T))
            logits = logits_flat.view(B, num_crops, 2).mean(dim=1)  # average crops
            all_logits.append(logits.cpu()); all_labels.append(labels.cpu())

    metrics = compute_all_metrics(torch.cat(all_logits), torch.cat(all_labels))
    print(f"Epoch {epoch}: EER={metrics['eer']:.4f}, minDCF={metrics['min_dcf']:.4f}")

    if metrics['min_dcf'] < best_metric:
        best_metric = metrics['min_dcf']
        torch.save({'model_state_dict': model.state_dict()}, 'best_model.pt')

    if early_stopping(metrics['min_dcf']):
        print(f"Early stopping at epoch {epoch}")
        break

    if epoch > warmup_epochs:
        scheduler.step()
```

After training, the best checkpoint is reloaded and evaluated on Train/Dev/Eval, and everything is written under `save_dir` (see Section 7). Each system in the paper is trained under three seeds (42, 63, 2026) and reported as mean ± std.

## 6. Evaluation Pipeline

### 6.1 Overview

The evaluation utilities in `Model/utils.py` compute the full metric suite (EER, minDCF, actDCF, CLLR, accuracy, F1, recall, AUC-ROC) and optionally apply **Platt calibration + prior correction** for score normalization.

```python
from utils import evaluate_model, apply_platt_calibration, compute_metrics_from_scores

# Get predictions (TTA averages logits over crops)
dev_logits, dev_labels   = evaluate_model(model, dev_loader, device, use_tta=True)
eval_logits, eval_labels = evaluate_model(model, eval_loader, device, use_tta=True)

# Fit calibration on Dev, apply to Eval
calibrated_scores, _ = apply_platt_calibration(
    dev_logits.numpy(), dev_labels.numpy(), eval_logits.numpy()
)

# Compute metrics from calibrated probability scores
metrics = compute_metrics_from_scores(calibrated_scores, eval_labels.numpy())
print(f"EER: {metrics['eer']:.4f}, minDCF: {metrics['min_dcf']:.4f}, CLLR: {metrics['cllr']:.4f}")
```

### 6.2 ASVspoof5 Track 1 Metrics

The DCF metrics follow the ASVspoof5 Track 1 spec with `C_miss=1.0`, `C_fa=10.0`, `π_spf=0.05`, using the **normalized** form `DCF'(t) = β · P_miss(t) + P_fa(t)` where `β = C_miss·(1-π_spf) / (C_fa·π_spf) ≈ 1.90`.

**minDCF — minimum normalized DCF over all thresholds:**
```python
def compute_min_dcf(scores, labels, c_miss=1.0, c_fa=10.0, pi_spf=0.05):
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr

    beta = (c_miss * (1 - pi_spf)) / (c_fa * pi_spf)   # ≈ 1.90
    dcf_normalized = beta * fnr + fpr                  # DCF'(t)

    idx = np.argmin(dcf_normalized)
    return dcf_normalized[idx], thresholds[idx]        # (min_dcf, threshold)
```

**actDCF — normalized DCF at the Bayes-optimal threshold `τ = -log(β)`:**
```python
def compute_act_dcf(scores, labels, c_miss=1.0, c_fa=10.0, pi_spf=0.05):
    beta = (c_miss * (1 - pi_spf)) / (c_fa * pi_spf)

    # scores are probabilities P(bonafide|x); convert to log-likelihood ratios
    eps = 1e-10
    s = np.clip(scores, eps, 1 - eps)
    llr = np.log(s / (1 - s))

    # decide at the Bayes threshold
    predictions = (llr >= -np.log(beta)).astype(int)
    tp = np.sum((predictions == 1) & (labels == 1)); fn = np.sum((predictions == 0) & (labels == 1))
    fp = np.sum((predictions == 1) & (labels == 0)); tn = np.sum((predictions == 0) & (labels == 0))

    fnr = fn / (tp + fn + 1e-10)   # P_miss
    fpr = fp / (fp + tn + 1e-10)   # P_fa
    return beta * fnr + fpr        # normalized actDCF
```

**CLLR — calibration cost** (`compute_cllr`): `1/(2·log2) · [mean_bonafide log(1+e^-s) + mean_spoof log(1+e^s)]`, with scores interpreted as log-likelihood ratios. Lower is better; 0 is perfect.

### 6.3 Complete Evaluation with Calibration

`evaluate_with_calibration` runs the whole pipeline: it evaluates Train (no TTA), Dev, and Eval (both with TTA), then — if `apply_calibration=True` — fits Platt calibration on **Dev** and applies it (with optional prior correction) to Dev and Eval.

```python
from utils import evaluate_with_calibration

results = evaluate_with_calibration(
    model=model,
    train_loader=train_loader,
    dev_loader=dev_loader,            # TTA loaders
    eval_loader=eval_loader,
    device=device,
    apply_calibration=True,          # fit on Dev, apply to Dev/Eval
    enable_prior_correction=True     # correct for class-prior mismatch
)

# results[split] has: 'initial_metrics' (uncalibrated) and, when calibration is on,
# 'calibrated_metrics'. split is one of 'train', 'dev', 'eval'.
print(results['eval']['initial_metrics'])
print(results['eval']['calibrated_metrics'])
```

> `main_train.py` calls this with `apply_calibration=False` for the final report (raw logits), while the calibration path is available for producing calibrated submission scores. To evaluate an already-saved checkpoint, use `Model/read_and_evaluate.py`, which rebuilds the model from its JSON config (see Section 7.2).

## 7. Project Structure

```
TFPARN/
│
├── Model/                          # Main TFPARN model + the two baselines
│   │
│   ├── model.py                    # Transformer model architecture
│   │   ├── SpeechClassifierArgs           # Model configuration dataclass
│   │   ├── PositionalEncoding             # Sinusoidal positional encoding
│   │   ├── SpeechTransformerClassifier    # Main model (in-model mel + Transformer)
│   │   └── create_model()                 # Model factory function
│   │
│   ├── data_process.py             # Data loading and preprocessing
│   │   ├── DefaultArgs                     # Data loading configuration
│   │   ├── RawBoost                        # RawBoost augmentation (3 algorithms)
│   │   ├── read_protocol()                 # Parse ASVspoof5 protocol files
│   │   ├── ASV5Dataset                     # PyTorch Dataset (FLAC -> fixed-length waveform)
│   │   ├── TTADataset                      # Test-Time Augmentation wrapper
│   │   ├── collate_fn() / collate_fn_tta() # Batch collation
│   │   └── make_loaders()                  # Train/Dev/Eval DataLoader creation
│   │
│   ├── utils.py                    # Utilities: losses, metrics, calibration, checkpointing
│   │   ├── set_seed() / get_device()       # Reproducibility & device
│   │   ├── FocalLoss / PairwiseRankingLoss / CombinedLoss
│   │   ├── create_loss_function()          # Build the (combined) loss
│   │   ├── compute_eer() / compute_min_dcf() / compute_act_dcf() / compute_cllr()
│   │   ├── compute_all_metrics()           # All metrics from logits
│   │   ├── apply_platt_calibration() / apply_prior_correction()
│   │   ├── evaluate_model() / evaluate_with_calibration()
│   │   ├── load_model_weights() / save_model()
│   │   └── EarlyStopping                   # Early stopping handler
│   │
│   ├── main_train.py               # End-to-end training pipeline
│   │   ├── ModelArgs                       # Complete training configuration
│   │   ├── train_one_epoch()               # One training epoch
│   │   ├── validate()                      # Validation with TTA
│   │   └── main()                          # Train -> select best -> evaluate -> save
│   │
│   ├── read_and_evaluate.py        # Evaluate a saved checkpoint
│   │   ├── EvaluationConfig                # Paths + TTA setting
│   │   ├── load_config_from_json()         # Reconstruct architecture from JSON
│   │   ├── evaluate_single_set()           # Evaluate one dataset
│   │   └── evaluate_model()                # Run on Train/Dev/Eval
│   │
│   ├── run_multiple_experiments.py # Run several ModelArgs configs back-to-back
│   │   ├── create_experiment_list()        # Define the experiments
│   │   └── run_single_experiment()         # Train + evaluate + save one config
│   │
│   ├── requirements.txt            # Training / evaluation dependencies (PyTorch + CUDA 13)
│   │
│   └── Baselines/                  # Re-implemented comparison baselines
│       ├── AASIST/                 # SincConv + graph attention (AASIST / AASISTArgs)
│       │   ├── model.py  data_process.py  main_train.py  read_and_evaluate.py  utils.py
│       └── RawNet2/                # SincNet front-end + GRU (RawNet2 / RawNet2Args)
│           ├── model.py  data_process.py  main_train.py  read_and_evaluate.py  utils.py
│
├── CsvAnalyzer/                    # Training-efficiency analysis & plotting
│   ├── csv_analyzer.py             # Single-model report (1 model x 3 seeds)
│   ├── csv_compare.py              # Cross-model comparison (N models x 3 seeds)
│   ├── csv_common.py               # Shared loading / interpolation / plotting / stats
│   ├── PLOTTING_METHOD.md          # Detailed description of the plotting method
│   └── requirements.txt            # Plotting dependencies (matplotlib, SciencePlots, Jupyter)
│
├── Introduction_of_TFPARN.ipynb    # This notebook
├── README.md                       # Quick-start guide
└── LICENSE
```

> Each model directory (`Model/`, `Model/Baselines/AASIST/`, `Model/Baselines/RawNet2/`) is self-contained and uses flat relative imports. Run scripts from *inside* the directory that contains them, e.g. `cd Model && python main_train.py`.

### 7.1 Core Modules (`Model/`)

**model.py** — the complete Transformer architecture with in-model mel spectrogram computation and flexible pooling.

**data_process.py** — ASVspoof5 protocol parsing, FLAC loading, fixed-length crop/repeat to 4 s, RawBoost augmentation (training only), and TTA (dev/eval).

**utils.py** — Focal + Pairwise ranking losses, the full metric suite (EER, minDCF, actDCF, CLLR, accuracy, F1, recall, AUC-ROC), Platt calibration + prior correction, checkpoint I/O, and early stopping.

### 7.2 Scripts (`Model/`)

**main_train.py** — automatic device selection, training with early stopping (by minDCF on Dev), best-model selection, a final Train/Dev/Eval evaluation, and checkpoint saving.

**read_and_evaluate.py** — reloads a saved model **from its JSON config** so the architecture is reconstructed exactly, then reports metrics on Train/Dev/Eval (TTA on dev/eval).

**run_multiple_experiments.py** — runs a list of `ModelArgs` configs one after another, saving each like `main_train.py` and writing a combined `experiments_summary_<timestamp>.json`.

### 7.3 Training Outputs

`main_train.py` (and each experiment) writes, under `save_dir`:

```
save_dir/
├── best_model.pt                                   # rolling best checkpoint during training
└── best_model_<metric>_<value>/
    ├── best_model_<metric>_<value>.pt              # final weights + optimizer state
    ├── best_model_<metric>_<value>.json            # full config (data/model/train args) + metrics
    └── timing_records.csv                           # per-run time, memory & metrics
```

`timing_records.csv` has columns `run_num, type, time, max_memory, if_best, eer, min_dcf, cllr, act_dcf`, where `type` is `1`=Train, `2`=Dev, `3`=final Eval. This file is exactly what `CsvAnalyzer/` consumes (Section 9).

### 7.4 Baselines (`Model/Baselines/`)

Both baselines reuse the same `data_process.py` pipeline, metrics, and training/evaluation flow as TFPARN — only the architecture differs:

- **AASIST** — SincConv front-end, 2D residual blocks, and homogeneous/heterogeneous graph-attention layers with graph pooling.
- **RawNet2** — SincNet convolution front-end, 1D residual blocks with per-block attention, and a 3-layer GRU.

### 7.5 Analysis (`CsvAnalyzer/`)

Turns the `timing_records.csv` logs into time-vs-metric curves and a `summary.md`. `csv_analyzer.py` reports a single model; `csv_compare.py` overlays several models. Both share `csv_common.py`. See `PLOTTING_METHOD.md` and Section 9.

## 8. Technical Summary

### 8.1 Key Innovations

**1. Focal Loss for Hard-Trial Emphasis**
- Down-weights easy, well-classified examples and concentrates gradient on hard, near-boundary trials
- Focal Loss: `FL(p_t) = -α_t * (1 - p_t)^γ * log(p_t)`
- Uses **equal class weights (α=0.5)** — it is deliberately *not* used to rebalance the imbalanced data, only to emphasize hard samples
- In the ablation it yields the largest single improvement in the calibration-dependent Cllr
- Hyperparameters: α=0.5, γ=2.0

**2. Pairwise Ranking Loss**
- Directly targets ranking-based metrics (EER/minDCF)
- Encourages bonafide samples to score higher than spoof samples
- Loss: `L_pairwise = max(0, margin - (score_bonafide - score_spoof))`, over all (bonafide, spoof) pairs in the batch
- Combined with the main loss: `L_total = L_main + λ * L_pairwise`
- Hyperparameters: margin=1.0, λ=0.3 (acts mainly on decision cost / score ordering, not the equal-error point)

**3. Attention Pooling**
- Learned attention mechanism for frame-level aggregation
- Automatically weights informative frames higher than uninformative frames
- More effective than mean pooling for the local, sparse spoofing cues
- Attention scores: `α_t = softmax(w^T * tanh(W * h_t))`
- Aggregated representation: `h = Σ α_t * h_t`; adds only ~33k parameters

**4. Test-Time Augmentation**
- Generate 5 overlapping crops per sample during inference
- Average logits across crops for robust predictions
- Reduces prediction variance caused by random cropping

**5. RawBoost Augmentation**
- Three augmentation algorithms (convolution, filtering, noise)
- Applied during training only, with probability `rawboost_prob` (default 0.5)
- Improves generalization to unseen attacks and codec variations
- Prevents overfitting to the training data distribution

### 8.2 Architecture Highlights

**Transformer Configuration (TFPARN defaults):**
```
Input: [B, 1, 64000] → 16kHz, 4-second waveform
Frontend: Log-Mel Spectrogram [B, T'≈401, 160]   (n_mels=160, n_fft=1024, hop=160)
Embedding: LayerNorm(freq) + Linear projection [B, 401, 256]
Positional: Sinusoidal encoding
Backbone: 6-layer Transformer (8 heads, d_model=256, dim_feedforward=1024)
Pooling: Mean / Attention / Top-k (Attention is used in TFPARN)
Classifier: 2-layer MLP → [B, 2] logits  (spoof=0, bonafide=1)
```

**Model Size:**
- Parameters: 4,846,275 (≈4.85M) with attention pooling; 4,813,250 (≈4.81M) with mean pooling
- Peak inference memory: **1.4 GB** (the lowest among the compared systems)

**Efficiency (from the paper, mean over 3 seeds):**
- ~135 s per training epoch (batch size 64)
- ~0.79 ms per-utterance inference latency (Dev, with TTA)
- Full model reaches its best Dev checkpoint in ~149 min — far less than AASIST (~1015 min)

### 8.3 Advantages and Limitations

**Advantages:**

✓ End-to-end trainable (no preprocessing)

✓ Best detection metrics among the compared systems, at the lowest inference memory

✓ Calibrated probability outputs available (Platt calibration + prior correction)

✓ Flexible pooling strategies

✓ TTA for improved robustness

✓ Better generalization (small train/dev gap) than the baselines


**Limitations:**

✗ Requires a GPU for efficient training (CPU works but is very slow)

✗ The pairwise hinge only loosely approximates minDCF/actDCF — a listwise/differentiable ranking loss could track the metrics more directly

✗ No explicit codec awareness

## 9. Baselines and Analysis Tools

### 9.1 Baseline Models (`Model/Baselines/`)

Two raw-waveform anti-spoofing baselines are provided for comparison. They **share TFPARN's `data_process.py` pipeline** (FLAC loading, 4 s fixed-length crop/repeat, RawBoost, TTA), the **same metric suite** (`compute_eer`, `compute_min_dcf`, `compute_act_dcf`, `compute_cllr`), and the **same train → select-best → evaluate → save flow** — only the model architecture differs. Each is self-contained; run it from inside its own folder after filling in the data/protocol/save paths.

**AASIST** (`AASIST` / `AASISTArgs` in `Model/Baselines/AASIST/model.py`)
- SincConv front-end (`first_conv=128`)
- Six 2D residual blocks (SELU activation)
- Homogeneous + heterogeneous **graph attention** layers (`gat_dims=[64, 32]`) with graph pooling (`pool_ratios=[0.5, 0.7, 0.5, 0.5]`)
- Training defaults: `batch_size=64`, `learning_rate=2.667e-4`, `max_epochs=100`, `loss_type="ce"`, `early_stopping_metric="min_dcf"`

```bash
cd Model/Baselines/AASIST
python main_train.py
```

**RawNet2** (`RawNet2` / `RawNet2Args` in `Model/Baselines/RawNet2/model.py`)
- SincNet convolution front-end (`sinc_out_channels=20`, `sinc_kernel_size=1024`)
- Six 1D residual blocks with per-block attention (LeakyReLU 0.3)
- 3-layer **GRU** (`gru_node=1024`, `nb_gru_layer=3`) + FC layers
- Training defaults: `batch_size=64`, `learning_rate=5e-5`, `max_epochs=100`, `loss_type="ce"`, `early_stopping_metric="min_dcf"`

```bash
cd Model/Baselines/RawNet2
python main_train.py
```

Both baselines produce the same checkpoint layout and `timing_records.csv` as TFPARN, so their logs feed straight into the CSV analyzer below.

### 9.2 Training-Efficiency Analysis (`CsvAnalyzer/`)

The CSV analyzer turns the `timing_records.csv` files into **time-vs-metric plots** (EER, minDCF, CLLR, actDCF) and a `summary.md`. The x-axis is **cumulative Train+Dev time** on a log scale (not the epoch index), so models with very different run lengths can be compared fairly. Inputs are given **per model as three seed CSVs** (seeds 42, 63, 2026); a `""` path marks a seed that wasn't run.

When a model has 2+ seeds, each seed's `(time, metric)` curve is interpolated onto a shared log-spaced grid and averaged (mean ± std band). Single-seed models plot their raw per-epoch points. Best-epoch "stars" (per metric) are taken from the raw CSV and drawn only on the Dev plots. See `CsvAnalyzer/PLOTTING_METHOD.md` for the full method and statistics.

```bash
pip install -r CsvAnalyzer/requirements.txt
cd CsvAnalyzer

# Single-model report: edit MODEL_NAME, SEED_CSV_PATHS, OUTPUT_DIR_PATH
python csv_analyzer.py

# Cross-model comparison: edit the MODEL_INPUTS list and OUTPUT_DIR_PATH
python csv_compare.py
```

Each run writes 8 PDFs (`{EER,Min_DCF,Cllr,actDCF}_per_{Train,Dev}.pdf`) plus `summary.md` into `OUTPUT_DIR_PATH`. Reported per-model statistics include memory usage, time per training epoch, total Train+Val time, time-to-best-model, per-utterance latency, and the best Dev checkpoint (selected by minDCF).

## 10. Experimental Results

All systems are trained and evaluated under one **unified protocol** (same 16 kHz / 4.0 s waveform handling, same batch size 64, three seeds 42/63/2026), so results isolate the **back-end architecture**. For the baselines, the Sinc-like front-end filterbanks are **kept but frozen (non-trainable)** so only the back-ends are optimized. TFPARN uses a log-Mel front-end (`n_fft=1024`, `n_mels=160`).

### 10.1 Systems and Ablation Design

| ID | System | Loss | Pairwise | Pooling |
|----|--------|------|----------|---------|
| 1 | AASIST   | CE   | No  | Graph pooling |
| 2 | RawNet2  | CE   | No  | Global max |
| 3 | TFPARN   | CE   | No  | Mean |
| 4 | TFPARN   | CE   | Yes | Mean |
| 5 | TFPARN   | Focal (α=0.5, γ=2.0) | Yes | Mean |
| 6 | TFPARN   | Focal (α=0.5, γ=2.0) | Yes | **Attention** |

The TFPARN ablation isolates each component: **3 → 4** adds the pairwise branch, **4 → 5** swaps CE → focal, **5 → 6** swaps mean → attention pooling. ID 6 is the full TFPARN.

### 10.2 Detection Metrics (mean ± std over 3 seeds)

| ID | EER (%) ↓ | minDCF ↓ | Cllr ↓ | actDCF ↓ |
|----|-----------|----------|--------|----------|
| 1 AASIST  | 18.58 ± 0.16 | 0.2911 ± 0.0026 | 2.6545 ± 0.8416 | 0.4966 ± 0.1648 |
| 2 RawNet2 | 27.23 ± 0.50 | 0.5375 ± 0.0052 | 2.8672 ± 0.2752 | 0.7214 ± 0.0742 |
| 3 TFPARN (CE, mean)      | 12.91 ± 0.09 | 0.2662 ± 0.0045 | 1.8796 ± 0.3314 | 0.3547 ± 0.0224 |
| 4 TFPARN (CE+pair, mean) | 12.92 ± 0.07 | 0.2561 ± 0.0021 | 1.6786 ± 0.3918 | 0.3160 ± 0.0526 |
| 5 TFPARN (focal+pair, mean) | 12.70 ± 0.36 | 0.2499 ± 0.0031 | **0.7232 ± 0.0787** | 0.3325 ± 0.0211 |
| 6 **TFPARN (full, attention)** | **12.52 ± 0.11** | **0.2430 ± 0.0043** | 0.9243 ± 0.4907 | **0.2897 ± 0.0191** |

**Reading the table:**
- **Even base TFPARN (ID 3)** — plain CE with mean pooling — already beats both baselines on *every* metric (EER 12.91% vs 18.58% / 27.23%; minDCF 0.2662 vs 0.2911 / 0.5375).
- **Pairwise (3 → 4):** EER essentially unchanged (12.91 → 12.92) but minDCF, Cllr and actDCF all drop — the ranking term acts on decision cost and score ordering, not the equal-error point.
- **Focal (4 → 5):** small EER/minDCF gains and a **large Cllr drop (1.6786 → 0.7232)** — the biggest single calibration improvement; since α=0.5 this reflects focusing on hard samples, not rebalancing.
- **Attention (5 → 6):** the lowest **minDCF (0.2430)**, **EER (12.52%)** and **actDCF (0.2897)** of any system. The primary **minDCF improves monotonically along the ablation chain: 0.2662 → 0.2561 → 0.2499 → 0.2430.**

The full model (ID 6) is best on three of four metrics including the primary minDCF; the focal variant (ID 5) gives the lowest, most stable Cllr. Both remain far ahead of the baselines.

### 10.3 Training & Inference Efficiency (mean ± std over 3 seeds)

| ID | Params | Inference mem (GB) ↓ | Time/epoch (s) ↓ | Time to best (min) ↓ | Latency (ms/utt) ↓ |
|----|--------|----------------------|------------------|----------------------|--------------------|
| 1 AASIST  | 0.30M  | 56.7 | 1289.4 ± 0.7 | 1014.61 ± 634.32 | 10.4805 ± 0.0034 |
| 2 RawNet2 | 17.62M | 4.9  | 94.5 ± 1.1   | 73.77 ± 27.63    | 0.7802 ± 0.0203 |
| 3 TFPARN  | 4.81M  | **1.4** | 136.3 ± 1.8 | 207.84 ± 59.17 | 0.8073 ± 0.0542 |
| 4 TFPARN  | 4.81M  | **1.4** | 134.4 ± 0.9 | 254.70 ± 52.87 | 0.7873 ± 0.0097 |
| 5 TFPARN  | 4.81M  | **1.4** | 134.9 ± 0.4 | 167.00 ± 22.66 | 0.7893 ± 0.0040 |
| 6 TFPARN  | 4.85M  | **1.4** | 135.6 ± 0.8 | **149.40 ± 39.51** | 0.7896 ± 0.0054 |

**Parameter count is a poor predictor of cost.** AASIST has the *fewest* parameters (0.30M) yet is by far the most expensive: 56.7 GB peak inference memory, ~1289 s/epoch, and 10.48 ms/utt (≈40× the memory, ≈10× the epoch time, and ≈13× the latency of TFPARN). RawNet2 is the *largest* model (17.62M) but cheap per epoch (94.5 s).

**Why AASIST is so heavy:** its graph back-end builds full pairwise-node tensors and repeats this across spectral and temporal branches, four heterogeneous graph-attention layers on two paths, and several top-k graph-pooling stages — all on feature maps from the 64,000-sample raw waveform. These irregular ops run inefficiently on GPUs. TFPARN instead uses standard dense operations (6 Transformer layers + small heads) on a short **401-frame** log-Mel sequence; self-attention is quadratic in length but 401 frames keep it cheap.

**Generalization, not just fitting.** The training-set curves show RawNet2 actually attains the *lowest training* minDCF/EER yet generalizes *worst* on Dev — clear overfitting. TFPARN keeps a much smaller train/dev gap, and it is the only system whose **calibration metrics (Cllr, actDCF) hold up on Dev** (AASIST's are highly unstable with wide seed spread; RawNet2's stay high). So TFPARN's advantage extends beyond ranking to the actual score scale.

> These curves are exactly the ones produced by `CsvAnalyzer/csv_compare.py` from the per-epoch `timing_records.csv` logs (Section 9.2): each metric vs. cumulative wall-clock training time on a log axis, with the across-seed mean line, a ±std band, and best-epoch stars on the Dev panels.

### 10.4 Takeaway

TFPARN **matches or surpasses both baselines on detection while training and running at substantially lower cost** — the lowest inference memory (1.4 GB), ~0.79 ms/utt latency, and the fastest convergence to its best Dev checkpoint among the variants. This is the paper's central efficiency claim: better detection *and* better efficiency, with the gains reflecting genuine generalization rather than overfitting.

## 11. References

### 11.1 This Work

- **A Training-Efficient Transformer-Based Anti-Spoofing Network for Logical Access in ASVspoof 5** — Sidan Yin (San Domenico School) and Bo Zhao (University of Washington). The paper that this notebook documents (`TFPARN.pdf` in the repository root).

### 11.2 Datasets

- ASVspoof 5 Evaluation Plan / data: https://www.asvspoof.org · https://zenodo.org/records/14498691
- Multilingual LibriSpeech (MLS): "MLS: A Large-Scale Multilingual Dataset for Speech Research" (Interspeech 2020)
- ASVspoof 2021 Dataset: https://www.kaggle.com/datasets/mohammedabdeldayem/avsspoof-2021
- ASVspoof 2019 Database: https://www.kaggle.com/datasets/awsaf49/asvpoof-2019-dataset

### 11.3 Methods

1. **Attention Is All You Need** (NeurIPS 2017) — the Transformer backbone. https://arxiv.org/abs/1706.03762
2. **Focal Loss for Dense Object Detection** (ICCV 2017) — hard-sample emphasis. https://arxiv.org/abs/1708.02002
3. **Pairwise Discriminative Speaker Verification in the I-vector Space** (IEEE/ACM TASLP 2013) — the pairwise ranking objective.
4. **Attentive Statistics Pooling for Deep Speaker Embedding** (Interspeech 2018) — the attention-pooling formulation.
5. **RawBoost** (ICASSP 2022) — three waveform augmentation algorithms (implemented in `data_process.py`). https://arxiv.org/abs/2111.04433
6. **Decoupled Weight Decay Regularization (AdamW)** (ICLR 2019) and **SGDR: warm restarts / cosine annealing** (ICLR 2017) — optimizer and LR schedule.
7. **Accurate, Large Minibatch SGD** (2017) — the linear learning-rate scaling rule. https://arxiv.org/abs/1706.02677
8. **Platt Calibration** (1999) — sigmoid-based probability calibration. https://www.cs.colorado.edu/~mozer/Teaching/syllabi/6622/papers/Platt1999.pdf

### 11.4 Baseline Models

9. **AASIST: Audio Anti-Spoofing using Integrated Spectro-Temporal Graph Attention Networks** (ICASSP 2022). https://arxiv.org/abs/2110.01200
10. **End-to-End anti-spoofing with RawNet2** (ICASSP 2021). https://arxiv.org/abs/2011.01108

### 11.5 Metrics and Evaluation

**EER (Equal Error Rate):** the point where FPR = FNR. Lower is better.

**minDCF / actDCF (Detection Cost Function):** weighted miss / false-alarm cost (ASVspoof5 Track 1: `C_miss=1.0`, `C_fa=10.0`, `π_spf=0.05`). `minDCF` = minimum normalized DCF over all thresholds; `actDCF` = normalized DCF at the Bayes threshold `τ = -log(β)`. Lower is better.

**Cllr (Cost of Log-Likelihood Ratio):** calibration quality (0 = perfect). "Application-Independent Evaluation of Speaker Detection" (Computer Speech & Language 2006). https://www.sciencedirect.com/science/article/abs/pii/S0885230805000306